In [12]:
import torch
from synthetic_images import visualize_n_samples

X = torch.load("./samples/diffusion_samples.pt")   
visualize_n_samples(
    X_train=X,
    y_train=None,            
    n=44,                    
    output_binarization=False,
    file_name="diffusion_samples.pdf"
)

print("Saved to plots/diffusion_samples.pdf")


Saved to plots/diffusion_samples.pdf


In [10]:
import numpy as np

def soft_path_score(image, shape=(28, 28), threshold=0.5):
    # works on different shape
    img = np.array(image)
    if img.ndim == 3:
        img = img[0]
    assert img.shape == shape

    # get coordinates of white pixels
    mask = img > threshold
    ys, xs = np.where(mask)
    n = len(xs)
    if n == 0:
        return 0.0, 0  

    # number the graph nodes
    coords = list(zip(ys, xs))
    idx = {c: i for i, c in enumerate(coords)}

    # get adjacency list
    adj = [[] for _ in range(n)]
    deg = np.zeros(n, dtype=int)
    for i, (y, x) in enumerate(coords):
        for dy, dx in [(-1,0), (1,0), (0,-1), (0,1)]:
            nb = (y + dy, x + dx)
            j = idx.get(nb)
            if j is not None:
                adj[i].append(j)
                deg[i] += 1

    # all pixels have degree at most two, degree <= 2
    good_deg = np.sum(deg <= 2)
    degree_score = good_deg / n          # 1 if no branching, else < 1 if many deg>2

    # there is exactly one connected component
    visited = np.zeros(n, dtype=bool)
    comp_sizes = []

    for start in range(n):
        if visited[start]:
            continue
        stack = [start]
        visited[start] = True
        size = 0
        while stack:
            i = stack.pop()
            size += 1
            for j in adj[i]:
                if not visited[j]:
                    visited[j] = True
                    stack.append(j)
        comp_sizes.append(size)

    largest_cc = max(comp_sizes)
    connectivity_score = largest_cc / n  # 1 if single component

    # the endpoint degree pattern matches that of a self-avoiding path
    num_deg0 = np.sum(deg == 0)
    num_deg1 = np.sum(deg == 1)

    if n == 1:
        # ideal: exactly one isolated pixel
        deviation = abs(num_deg0 - 1) + num_deg1  
    else:
        # ideal: two endpoints (deg=1) and no deg=0
        deviation = abs(num_deg1 - 2) + max(0, num_deg0) 

    endpoint_score = 1.0 - min(1.0, deviation / n)

    # combine 
    score = degree_score * connectivity_score * endpoint_score

    return float(score), int(n)

def soft_validity_score(images, shape=(28, 28), threshold=0.5):
    imgs = np.array(images)
    if imgs.ndim == 4:
        imgs = imgs[:, 0]

    batch = imgs.shape[0]
    scores = []
    lengths = []

    for i in range(batch):
        score, length = soft_path_score(imgs[i], shape=shape, threshold=threshold)
        scores.append(score)
        lengths.append(length)

    scores = np.array(scores)
    lengths = np.array(lengths)
    mean_score = float(scores.mean())

    return mean_score, lengths, scores

Soft validity score (mean): 0.3079472007363181
First 10 per-image scores: [0.29472007 0.20882969 0.27758588 0.32720872 0.22154375 0.17566667
 0.18144663 0.35009183 0.20952899 0.29213676 0.35595703 0.
 0.340032   0.24970273 0.39795007 0.25550964 0.33518029 0.24489796
 0.27793629 0.13803597 0.21700527 0.18309542 0.33619444 0.13177615
 0.51068971 0.28526484 0.56319223 0.16184051 0.30828134 0.28973293
 0.39543773 0.19848771 0.513      0.3682304  0.34545726 0.31415344
 0.30723714 0.52224    0.21417667 0.32142101 0.55169487 0.35803999
 0.62014815 0.24813291 0.37144387 0.46875    0.13718824 0.09670921
 0.25771094 0.66666667]


In [15]:
import torch

data = torch.load("./syntetic_data_normal_train.pt")
X_train = data["images"]      
y_train = data["labels"]      

mean_score, lengths, scores = soft_validity_score(X_train.numpy(), threshold=0.5)

print("Soft validity score (mean):", mean_score)
print("Validity score on synthetic train set:", scores)
print("First 10 valid path lengths:", lengths[:10])

Soft validity score (mean): 1.0
Validity score on synthetic train set: [1. 1. 1. ... 1. 1. 1.]
First 10 valid path lengths: [  4  79  86 137  45  68  74  40  77 102]


In [16]:
X_gen = torch.load("./samples/diffusion_samples.pt")  
mean_score, lengths, scores = soft_validity_score(X_gen.numpy(), threshold=0.5)

print("Soft validity score (mean):", mean_score)
print("First 10 per-image scores:", scores)

Soft validity score (mean): 0.3079472007363181
First 10 per-image scores: [0.29472007 0.20882969 0.27758588 0.32720872 0.22154375 0.17566667
 0.18144663 0.35009183 0.20952899 0.29213676 0.35595703 0.
 0.340032   0.24970273 0.39795007 0.25550964 0.33518029 0.24489796
 0.27793629 0.13803597 0.21700527 0.18309542 0.33619444 0.13177615
 0.51068971 0.28526484 0.56319223 0.16184051 0.30828134 0.28973293
 0.39543773 0.19848771 0.513      0.3682304  0.34545726 0.31415344
 0.30723714 0.52224    0.21417667 0.32142101 0.55169487 0.35803999
 0.62014815 0.24813291 0.37144387 0.46875    0.13718824 0.09670921
 0.25771094 0.66666667]
